# 🚀 Getting Started with the Data Science Portfolio

Welcome to the Data Science Portfolio! This tutorial will walk you through the basics of using our ML Pipeline, Statistical Methods, and Dashboard modules.

## 📋 Prerequisites

Before we begin, make sure you have:
- Python 3.8 or higher installed
- All required packages installed (see requirements.txt)
- Access to the portfolio codebase

## 🎯 Learning Objectives

By the end of this tutorial, you will:
1. Load and explore data
2. Perform basic statistical analysis
3. Build a simple ML model
4. Create interactive visualizations
5. Deploy a basic dashboard

## 1️⃣ Setup and Imports

In [ ]:
# Standard imports
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

# Portfolio imports
from dashboard_framework import DashboardConfig, EnhancedDashboard
from modern_bank_churn.ml_pipeline_orchestrator import (
    MLPipelineOrchestrator,
    PipelineConfig,
)
from statistical_methods.hypothesis_tester import HypothesisTester
from statistical_methods.statistical_analyzer import StatisticalAnalyzer
from visualization_components import InteractiveVisualizations

# Set style
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

print("✅ All modules imported successfully!")

## 2️⃣ Load Sample Data

Let's start by loading a sample dataset. We'll use a synthetic customer churn dataset for this tutorial.

In [ ]:
# Generate sample data
np.random.seed(42)
n_samples = 1000

data = pd.DataFrame(
    {
        "customer_id": range(1, n_samples + 1),
        "age": np.random.normal(45, 15, n_samples).astype(int),
        "tenure": np.random.exponential(24, n_samples),
        "balance": np.random.lognormal(10, 1.5, n_samples),
        "num_products": np.random.choice(
            [1, 2, 3, 4], n_samples, p=[0.5, 0.3, 0.15, 0.05]
        ),
        "credit_score": np.random.normal(650, 100, n_samples),
        "is_active": np.random.choice([0, 1], n_samples, p=[0.3, 0.7]),
        "salary": np.random.lognormal(11, 0.5, n_samples),
        "churn": np.random.choice([0, 1], n_samples, p=[0.8, 0.2]),
    }
)

# Add categorical variables
data["geography"] = np.random.choice(
    ["USA", "UK", "Germany"], n_samples, p=[0.5, 0.3, 0.2]
)
data["gender"] = np.random.choice(["Male", "Female"], n_samples, p=[0.55, 0.45])

print("📊 Dataset Info:")
print(f"Shape: {data.shape}")
print(f"Churn rate: {data['churn'].mean():.2%}")
print("\n🔍 First few rows:")
data.head()

## 3️⃣ Statistical Analysis

Let's perform comprehensive statistical analysis using our StatisticalAnalyzer.

In [ ]:
# Initialize statistical analyzer
analyzer = StatisticalAnalyzer(data=data)

# Generate summary statistics
summary = analyzer.generate_summary(include_plots=False)

print("📈 Summary Statistics:")
for col in ["age", "tenure", "balance", "credit_score"]:
    if col in summary:
        stats = summary[col]
        print(f"\n{col.title()}:")
        print(f"  Mean: {stats['mean']:.2f}")
        print(f"  Std: {stats['std']:.2f}")
        print(f"  Median: {stats['median']:.2f}")
        print(f"  IQR: [{stats['q1']:.2f}, {stats['q3']:.2f}]")

In [ ]:
# Test for distribution
dist_results = analyzer.distribution_analysis(
    column="balance", test_distributions=["normal", "lognormal", "exponential"]
)

print("🔬 Distribution Analysis for 'balance':")
print(f"Best fitting distribution: {dist_results['best_fit']}")
print(f"p-value: {dist_results['p_value']:.4f}")

# Visualize distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
ax1.hist(data["balance"], bins=30, alpha=0.7, color="blue", edgecolor="black")
ax1.set_title("Balance Distribution")
ax1.set_xlabel("Balance")
ax1.set_ylabel("Frequency")

# Q-Q plot
from scipy import stats

stats.probplot(np.log(data["balance"]), dist="norm", plot=ax2)
ax2.set_title("Q-Q Plot (Log-transformed)")

plt.tight_layout()
plt.show()

## 4️⃣ Hypothesis Testing

Let's test if there's a significant difference in balance between churned and non-churned customers.

In [ ]:
# Initialize hypothesis tester
tester = HypothesisTester(alpha=0.05)

# Split data by churn status
churned = data[data["churn"] == 1]["balance"]
not_churned = data[data["churn"] == 0]["balance"]

# Perform t-test
result = tester.t_test(churned, not_churned, alternative="two-sided")

print("🎯 T-Test Results:")
print(f"T-statistic: {result['t_statistic']:.4f}")
print(f"P-value: {result['p_value']:.4f}")
print(f"Cohen's d: {result['cohens_d']:.3f}")

if result["p_value"] < 0.05:
    print("✅ Significant difference found!")
else:
    print("❌ No significant difference found.")

# Visualize the comparison
fig, ax = plt.subplots(figsize=(8, 6))
data.boxplot(column="balance", by="churn", ax=ax)
ax.set_title("Balance by Churn Status")
ax.set_xlabel("Churn (0=No, 1=Yes)")
ax.set_ylabel("Balance")
plt.suptitle("")  # Remove default title
plt.show()

## 5️⃣ Machine Learning Pipeline

Now let's build a machine learning model to predict customer churn.

In [ ]:
# Prepare data for ML
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Encode categorical variables
le = LabelEncoder()
data_ml = data.copy()
data_ml["geography_encoded"] = le.fit_transform(data_ml["geography"])
data_ml["gender_encoded"] = le.fit_transform(data_ml["gender"])

# Select features
feature_cols = [
    "age",
    "tenure",
    "balance",
    "num_products",
    "credit_score",
    "is_active",
    "salary",
    "geography_encoded",
    "gender_encoded",
]
X = data_ml[feature_cols]
y = data_ml["churn"]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("📊 Data Split:")
print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Class distribution (train): {y_train.value_counts(normalize=True).to_dict()}")

In [ ]:
# Configure and run ML pipeline
config = PipelineConfig(
    feature_selection_method="mutual_info",
    feature_selection_k=7,
    model_type="ensemble",
    hyperparameter_tuning=False,  # Skip for speed
    cross_validation_folds=3,
)

# Initialize pipeline
print("🔧 Initializing ML Pipeline...")
orchestrator = MLPipelineOrchestrator(config)

# Run pipeline
print("🚀 Training models...")
results = orchestrator.run_pipeline(pd.concat([X_train, y_train], axis=1))

print("\n✨ Pipeline Results:")
print(f"Best model: {results.best_model.__class__.__name__}")
print(f"Cross-validation AUC: {results.metrics.get('auc_roc', 0):.4f}")
print(f"Selected features: {results.selected_features[:5]}...")  # Show first 5

In [ ]:
# Evaluate on test set
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Make predictions
X_test_selected = X_test[results.selected_features]
y_pred = results.best_model.predict(X_test_selected)
y_pred_proba = results.best_model.predict_proba(X_test_selected)[:, 1]

# Calculate metrics
test_auc = roc_auc_score(y_test, y_pred_proba)

print("📊 Test Set Performance:")
print(f"AUC-ROC: {test_auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["No Churn", "Churn"]))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["No Churn", "Churn"],
    yticklabels=["No Churn", "Churn"],
)
plt.title("Confusion Matrix")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.show()

## 6️⃣ Interactive Visualizations

Let's create some interactive visualizations using our visualization components.

In [ ]:
# Initialize visualization component
viz = InteractiveVisualizations()

# Create 3D scatter plot
fig_3d = viz.create_interactive_3d_scatter(
    data,
    x_col="age",
    y_col="balance",
    z_col="tenure",
    color_col="churn",
    title="Customer Segmentation (3D)",
)

fig_3d.show()

print("💡 Tip: Drag to rotate the 3D plot!")

In [ ]:
# Create animated time series (simulated monthly data)
# Generate monthly aggregated data
months = pd.date_range("2023-01", periods=12, freq="M")
monthly_data = pd.DataFrame(
    {
        "month": months,
        "churn_rate": np.random.uniform(0.15, 0.25, 12),
        "avg_balance": np.random.uniform(20000, 30000, 12),
        "new_customers": np.random.randint(50, 150, 12),
    }
)

# Create animated chart
fig_animated = viz.create_animated_time_series(
    monthly_data,
    x_col="month",
    y_cols=["churn_rate", "avg_balance"],
    title="Monthly Metrics Trend",
    animation_speed=500,
)

fig_animated.show()

print("▶️ Click the play button to see the animation!")

## 7️⃣ Create a Simple Dashboard

Finally, let's create a simple dashboard to display our results.

In [ ]:
# Configure dashboard
config = DashboardConfig(
    app_name="Customer Churn Dashboard",
    port=8050,
    enable_export=True,
    enable_dark_mode=True,
    enable_filtering=True,
)

# Create dashboard instance
dashboard = EnhancedDashboard(config)


# Register data source
def get_dashboard_data(filters=None):
    df = data.copy()
    if filters:
        if "geography" in filters:
            df = df[df["geography"].isin(filters["geography"])]
        if "churn" in filters:
            df = df[df["churn"] == filters["churn"]]
    return df


dashboard.register_data_source("customer_data", get_dashboard_data)

# Add filters
dashboard.add_filter("geography", type="multi_select", options=["USA", "UK", "Germany"])
dashboard.add_filter(
    "churn", type="dropdown", options=[0, 1], labels=["No Churn", "Churn"]
)

# Add charts
dashboard.add_chart("churn_distribution", chart_type="pie")
dashboard.add_chart("age_distribution", chart_type="histogram")
dashboard.add_chart("balance_by_geography", chart_type="box")
dashboard.add_chart("correlation_heatmap", chart_type="heatmap")

print("🎨 Dashboard configured!")
print("\nTo run the dashboard, execute:")
print("dashboard.run()")
print("\nThen open http://localhost:8050 in your browser.")

## 🎓 Summary

Congratulations! You've completed the getting started tutorial. You've learned how to:

✅ Load and explore data  
✅ Perform statistical analysis  
✅ Conduct hypothesis testing  
✅ Build ML models using the pipeline  
✅ Create interactive visualizations  
✅ Set up a dashboard  

## 📚 Next Steps

1. **Explore Advanced Features**: Check out the advanced tutorials for more complex use cases
2. **Try Your Own Data**: Apply these techniques to your own datasets
3. **Customize the Pipeline**: Modify the pipeline configuration for your specific needs
4. **Build Production Dashboards**: Create more sophisticated dashboards with real-time data
5. **Read the API Documentation**: Dive deeper into the API reference for all available features

## 🔗 Useful Resources

- [API Reference](../docs/api_reference.md)
- [ML Pipeline Documentation](../docs/modules/ml_pipeline.md)
- [Statistical Methods Documentation](../docs/modules/statistics.md)
- [Dashboard Documentation](../docs/modules/dashboard.md)
- [GitHub Repository](https://github.com/your-repo)

Happy coding! 🚀